In [ ]:
import sys
import os
import tqdm
import matplotlib.pyplot as plt

build_dir = os.path.abspath('./build')
print(f"Build dir: {build_dir}")
print(f"Contents: {os.listdir(build_dir)}")  # ← CRITICAL DEBUG

sys.path.insert(0, build_dir)
if os.path.exists(build_dir):
    if os.name == 'nt':
        os.add_dll_directory(build_dir)

try:
    import tsn
    print("✅ SUCCESS!")
    print(dir(tsn))
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Trying direct import...")
    # List all .pyd files and try importing
    for f in os.listdir(build_dir):
        if f.endswith('.pyd'):
            modname = f[:-20]  # Remove .cp312-win_amd64.pyd
            print(f"Trying {modname}...")


import time
def check_overlap(messages,results):
    hyper_period = results.hyperperiod
    time = [dict() for i in range(hyper_period)]

    for msg in range(len(results.amount_sent)):
        for rep in range(len(results.amount_sent[msg])):
            for i in range(len(results.R[(msg,rep)])):

                route = results.R[(msg,rep)][i]
                start_time = results.departure_times[(msg,rep)][i]
                for idx in range(len(route) - 1):
                    u = min(route[idx],route[idx+1])
                    v = max(route[idx],route[idx+1])

                    for busy in range(messages[msg].size):
                        if(start_time + busy < hyper_period):
                            time[start_time + busy][(u,v)] = time[start_time + busy].get((u,v),0) + 1
                    start_time+=1

    flag = 1
    for d in time:
        for x,y in d.items():
            if(y > 1): 
                flag = 0

    return flag

def successfulPercetage(messages,results):
    fail = 0
    total = 0
    for msg in range(len(results.reps)):
        total += 1
        for rep in range(results.reps[msg]):
            if results.amount_sent[msg][rep] != messages[msg].tl:
                fail += 1
                break
    return 1 - fail/total

In [ ]:
memory_reserve = bytearray(1024**3)

In [ ]:
import matplotlib.pyplot as plt
import time

def run_experiment(assignment_type):
    message_counts = [10, 25, 40, 55, 70, 85, 100, 115]
    
    results = {
        "success_rate": [],
        "avg_cost": [],
        "avg_time": []
    }

    for num_messages in message_counts:
        print(f"Testing Assignment Type {assignment_type} | Num Messages: {num_messages}",flush=True)
        total_success = 0
        total_cost = 0
        total_time = 0
        
        runs = 10
        for seed in range(runs):
            # print("M IN",flush= True)
            M = tsn.makeInputs(num_ecu, num_bridges, num_messages, base_period, 
                               period_choice, period_choice_weights, 
                               min_size, max_size, min_tl, max_tl, seed)
            # print("M OUT",flush= True)

            start = time.time()
            output = tsn.algo(num_ecu, num_bridges, M, bridge_limit, link_build_cost, assignment_type, debug_print = 0)
            end = time.time()

            total_success += successfulPercetage(M, output)
            total_cost += output.Cost
            total_time += (end - start)

        results["success_rate"].append(total_success / runs)
        results["avg_cost"].append(total_cost / runs)
        results["avg_time"].append(total_time / runs)
        
    return message_counts, results

# --- Setup Parameters ---
num_ecu, num_bridges = 4, 2
bridge_limit, link_build_cost = 3, 2
base_period = 1
period_choice = tsn.VectorInt([7, 14, 28])
period_choice_weights = tsn.Vectordouble([0.1, 0.45, 0.45])
min_size, max_size = 2,5
min_tl, max_tl = 2, 2

# --- Run Experiments ---
# x_axis, data_0 = run_experiment(0)
x_axis, data_1 = run_experiment(1)

# Proof of Fragmentation

In [ ]:
import matplotlib.pyplot as plt
num_ecu = 2
num_bridges = 0
bridge_limit = 1
link_build_cost = 2
assignment_type = 0

M = tsn.VectorMessage([tsn.Message(0,1,3,5,1),tsn.Message(0,1,3,10,1)])
output = tsn.algo(num_ecu,num_bridges,M,bridge_limit,link_build_cost,assignment_type,0)

success = successfulPercetage(M,output)
print(success)

In [ ]:
import matplotlib.pyplot as plt
num_ecu = 2
num_bridges = 0
bridge_limit = 1
link_build_cost = 2
assignment_type = 1

M = tsn.VectorMessage([tsn.Message(0,1,3,5,1),tsn.Message(0,1,3,10,1)])
output = tsn.algo(num_ecu,num_bridges,M,bridge_limit,link_build_cost,assignment_type,0)

success = successfulPercetage(M,output)
print(success)

# Experimentaion

In [ ]:
import matplotlib.pyplot as plt
import time

def run_experiment(assignment_type):
    message_counts = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
    
    results = {
        "success_rate": [],
        "avg_cost": [],
        "avg_time": []
    }

    for num_messages in message_counts:
        print(f"Testing Assignment Type {assignment_type} | Num Messages: {num_messages}")
        total_success = 0
        total_cost = 0
        total_time = 0
        
        runs = 100
        for seed in range(runs):
            M = tsn.makeInputs(num_ecu, num_bridges, num_messages, base_period, 
                               period_choice, period_choice_weights, 
                               min_size, max_size, min_tl, max_tl, seed)

            start = time.time()
            output = tsn.algo(num_ecu, num_bridges, M, bridge_limit, link_build_cost, assignment_type)
            end = time.time()

            total_success += successfulPercetage(M, output)
            total_cost += output.Cost
            total_time += (end - start)

        results["success_rate"].append(total_success / runs)
        results["avg_cost"].append(total_cost / runs)
        results["avg_time"].append(total_time / runs)
        
    return message_counts, results

# --- Setup Parameters ---
num_ecu, num_bridges = 8, 4
bridge_limit, link_build_cost = 4, 2
base_period = 1
period_choice = tsn.VectorInt([5, 10])
period_choice_weights = tsn.Vectordouble([0.7, 0.3])
min_size, max_size = 3, 3
min_tl, max_tl = 2, 2

# --- Run Experiments ---
x_axis, data_0 = run_experiment(0)
_, data_1 = run_experiment(1)

# --- Plotting ---
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# 1. Success Rate
axs[0].plot(x_axis, data_0["success_rate"], 'o-', label='Type 0')
axs[0].plot(x_axis, data_1["success_rate"], 's--', label='Type 1')
axs[0].set_title('Success Rate vs Num Messages')
axs[0].set_xlabel('Num Messages')
axs[0].set_ylabel('Success %')
axs[0].legend()
axs[0].grid(True)

# 2. Average Topology Cost
axs[1].plot(x_axis, data_0["avg_cost"], 'o-', label='Type 0')
axs[1].plot(x_axis, data_1["avg_cost"], 's--', label='Type 1')
axs[1].set_title('Avg Topology Cost vs Num Messages')
axs[1].set_xlabel('Num Messages')
axs[1].set_ylabel('Cost')
axs[1].legend()
axs[1].grid(True)

# 3. Execution Time
axs[2].plot(x_axis, data_0["avg_time"], 'o-', label='Type 0')
axs[2].plot(x_axis, data_1["avg_time"], 's--', label='Type 1')
axs[2].set_title('Execution Time vs Num Messages')
axs[2].set_xlabel('Num Messages')
axs[2].set_ylabel('Seconds')
axs[2].legend()
axs[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import time

def run_experiment(assignment_type):
    message_counts = [10, 25, 40, 55, 70, 85, 100, 115]
    
    results = {
        "success_rate": [],
        "avg_cost": [],
        "avg_time": []
    }

    for num_messages in message_counts:
        print(f"Testing Assignment Type {assignment_type} | Num Messages: {num_messages}")
        total_success = 0
        total_cost = 0
        total_time = 0
        
        runs = 100
        for seed in range(runs):
            M = tsn.makeInputs(num_ecu, num_bridges, num_messages, base_period, 
                               period_choice, period_choice_weights, 
                               min_size, max_size, min_tl, max_tl, seed)

            start = time.time()
            output = tsn.algo(num_ecu, num_bridges, M, bridge_limit, link_build_cost, assignment_type)
            end = time.time()

            total_success += successfulPercetage(M, output)
            total_cost += output.Cost
            total_time += (end - start)

        results["success_rate"].append(total_success / runs)
        results["avg_cost"].append(total_cost / runs)
        results["avg_time"].append(total_time / runs)
        
    return message_counts, results

# --- Setup Parameters ---
num_ecu, num_bridges = 4, 2
bridge_limit, link_build_cost = 3, 2
base_period = 2
period_choice = tsn.VectorInt([7, 14, 28])
period_choice_weights = tsn.Vectordouble([0.1, 0.45, 0.45])
min_size, max_size = 6,8
min_tl, max_tl = 2, 2

# --- Run Experiments ---
x_axis, data_0 = run_experiment(0)
# _, data_1 = run_experiment(1)

# --- Plotting ---
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# 1. Success Rate
axs[0].plot(x_axis, data_0["success_rate"], 'o-', label='Type 0')
axs[0].plot(x_axis, data_1["success_rate"], 's--', label='Type 1')
axs[0].set_title('Success Rate vs Num Messages')
axs[0].set_xlabel('Num Messages')
axs[0].set_ylabel('Success %')
axs[0].legend()
axs[0].grid(True)

# 2. Average Topology Cost
axs[1].plot(x_axis, data_0["avg_cost"], 'o-', label='Type 0')
axs[1].plot(x_axis, data_1["avg_cost"], 's--', label='Type 1')
axs[1].set_title('Avg Topology Cost vs Num Messages')
axs[1].set_xlabel('Num Messages')
axs[1].set_ylabel('Cost')
axs[1].legend()
axs[1].grid(True)

# 3. Execution Time
axs[2].plot(x_axis, data_0["avg_time"], 'o-', label='Type 0')
axs[2].plot(x_axis, data_1["avg_time"], 's--', label='Type 1')
axs[2].set_title('Execution Time vs Num Messages')
axs[2].set_xlabel('Num Messages')
axs[2].set_ylabel('Seconds')
axs[2].legend()
axs[2].grid(True)

plt.tight_layout()
plt.show()